# ARIMA implementation

This notebook implements the statistical-model part of the experimental workflow described in the report. It uses the M4 Competition monthly subset, applies the complexity-based sampling strategy, compares seasonal and non-seasonal AutoARIMA by growing-window cross-validation, forecasts the 18-month test horizon, and stores the final errors used in the analysis.

**Report references:** Sections 3.1 (*Data*), 3.1.1 (*Stratified Pseudo-Random Selection Strategy*), and 3.2 (*Methodology, Models*).

## 1. Libraries and reproducibility

The report evaluates forecasting performance under a controlled experimental setup. This cell imports the libraries used for data loading, AutoARIMA modelling, stationarity diagnostics, metric computation, plotting, and parallel execution. The fixed seed supports reproducibility of the experiment

**Report reference:** Section 3.2 (*Methodology*).

In [2]:
%matplotlib inline

import re
import os
import random
from joblib import Parallel, delayed #enables parallel execution across CPU cores

import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning

from datasetsforecast.m4 import M4

import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, ARIMA, SeasonalNaive

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from functools import partial

import utilsforecast.losses as ufl
from utilsforecast.evaluation import evaluate

from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2. Loading the M4 Competition monthly data

The empirical study is based on monthly time series from the M4 Competition dataset.

**Report reference:** Section 3.1 (*Data*).

In [3]:
M4.download(directory="data", group="Monthly") 

df, *_ = M4.load(directory="../data", group="Monthly")

df.head()

,unique_id,ds,y
0,M1,1,8000.0
1,M1,2,8350.0
2,M1,3,8570.0
3,M1,4,7700.0
4,M1,5,7080.0


## 3. Stratified pseudo-random selection by Complexity Estimate

The initial sample is selected to cover the observed range of Complexity Estimate (CE). This cell computes CE for all monthly series, divides the full set into four CE equal-sized groups, and selects 100 series from each group, resulting in 400 series with balanced coverage across complexity levels.

**Report references:** Sections 2.1 (*Complexity Measures*) and 3.1.1 (*Stratified Pseudo-Random Selection Strategy*).

In [4]:
number_of_series_to_evaluate = 400
series_per_group = number_of_series_to_evaluate // 4

def complexity_estimate(ts):
    ts = np.asarray(ts)
    return np.sqrt(np.sum(np.diff(ts) ** 2))

#compute CE for all series
complexities = []

for uid, g in df.groupby("unique_id"):
    ts = g.sort_values("ds")["y"].values
    c = complexity_estimate(ts)

    complexities.append({
        "unique_id": uid,
        "complexity": c
    })

complexity_df = pd.DataFrame(complexities)

#create equal-sized groups
complexity_df["group"] = pd.qcut(
    complexity_df["complexity"],
    q=4,
    labels=[0, 1, 2, 3]
)

#order by unique_id on each group
complexity_df["uid_num"] = complexity_df["unique_id"].str[1:].astype(int)

complexity_df = (
    complexity_df
    .sort_values(["group", "uid_num"])
    .reset_index(drop=True)
)

#select series
series_ids = []
chunk_info = []

for q in range(4):
    group_df = (
        complexity_df[complexity_df["group"] == q]
        .reset_index(drop=True)
    )

    #divide each group in 100 segment
    chunks = np.array_split(group_df, series_per_group)

    for group_id, chunk in enumerate(chunks, start=1):
        chunk_info.append({
            "g": q,
            "group": group_id,
            "n_series_in_group": len(chunk)
        })

        #choose one series per segment
        if len(chunk) > 0:
            selected_id = chunk.sample(1, random_state=42)["unique_id"].iloc[0]
            series_ids.append(selected_id)

#number of selected series for each group
selected_counts_df = (
    complexity_df[complexity_df["unique_id"].isin(series_ids)]
    .groupby("group", observed=True)
    .size()
    .reset_index(name="n_selected")
)

print("\nGroup Sizes:")
print(selected_counts_df.to_string(index=False))

#order IDs
series_ids = sorted(
    series_ids,
    key=lambda x: int(x[1:])
)

df_selected = df[df["unique_id"].isin(series_ids)].copy()

#order for plots
df_sorted = (
    df_selected.assign(
        uid_num=df_selected["unique_id"].str[1:].astype(int)
    )
    .sort_values(["uid_num", "ds"])
    .drop(columns="uid_num")
)

grouped = {
    uid: g
    for uid, g in df_sorted.groupby("unique_id", sort=False)
}

df = df_selected.copy()

#plot selected series
n_cols = 4
n_rows = int(np.ceil(len(series_ids) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 2.5 * n_rows))
axes = axes.flatten()

for ax, s_id in zip(axes, series_ids):
    serie = grouped.get(s_id)

    if serie is not None:
        ax.plot(serie["ds"].values, serie["y"].values)

    ax.set_title(s_id, fontsize=8)
    ax.tick_params(labelsize=6)

for ax in axes[len(series_ids):]:
    ax.remove()

plt.tight_layout()
plt.savefig("series_plot.png", dpi=300, bbox_inches="tight")
plt.close()


Group Sizes:
group  n_selected
    0         100
    1         100
    2         100
    3         100


## 4. Compute the CE

The CE is the first complexity measure used in the report. This cell stores the CE value and length of each selected series in `final_df`, which becomes the central table for the later forecasting results.

**Report reference:** Section 2.1 (*Complexity Measures*), Equation 1.

In [15]:
complexity_df = pd.DataFrame(complexities)

complexity_df["complexity"] = complexity_df["complexity"].round(3)

final_df = (
    df_sorted
    .groupby("unique_id")["y"]
    .size()
    .reset_index(name="size")
)

final_df = final_df.merge(complexity_df, on="unique_id", how="left")

final_df = final_df.sort_values(
    by="unique_id",
    key=lambda col: col.str.extract(r"(\d+)")[0].astype(int)
).reset_index(drop=True)

final_df.head()

,unique_id,size,complexity
0,M80,585,22292.093
1,M160,91,273.198
2,M251,289,2168.225
3,M293,150,5295.457
4,M591,142,372.494


## 5. Stationarity helper

AutoARIMA modelling is constrained to a maximum of two differences to avoid excessive signal loss. This helper function applies ADF and KPSS tests and records the transformations required to make each series stationary, using the same differencing limit.

**Report reference:** Section 3.2 (*Models*).

In [16]:
def make_stationary(series, alpha=0.05, max_diff=2):
    
    series = series.dropna().copy()
    transformations = []
    status = None
    
    def adf_test(x):
        return adfuller(x, autolag='AIC')[1]
    
    def kpss_test(x):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", InterpolationWarning)
                return kpss(x, regression='ct', nlags='auto')[1]
        except:
            return np.nan
    
    for i in range(max_diff+1):
        
        adf_p = adf_test(series)
        kpss_p = kpss_test(series)
        
        adf_stat = adf_p <= alpha
        kpss_stat = kpss_p > alpha if not np.isnan(kpss_p) else None
        
        #stationary
        if adf_stat and kpss_stat:
            status = "stationary"
            return series, transformations, status

        if (i<=1):
            #non-stationary
            if not adf_stat and not kpss_stat:
                series = series.diff().dropna()
                transformations.append("difference")
            
            #trend-stationary
            elif not adf_stat and kpss_stat:
                t = np.arange(len(series))
                trend = np.polyfit(t, series, 1)
                trend_line = trend[0]*t + trend[1]
                
                series = series - trend_line
                transformations.append("detrend")
            
            #difference-stationary
            elif adf_stat and not kpss_stat:
                series = series.diff().dropna()
                transformations.append("difference")
            
            else:
                status = "inconclusive"
                return series, transformations, status
    
    status = "max_diff_reached"
    return series, transformations, status

## 6. Apply stationarity diagnostics to each series

The stationarity diagnostics are computed independently for each selected time series and merged into `final_df`. This preserves, for each instance, the transformation history and final stationarity status used to support the interpretation of the ARIMA workflow.

**Report reference:** Section 3.2 (*Models*).

In [17]:
grouped_series = {
    uid: g.sort_values("ds")["y"]
    for uid, g in df.groupby("unique_id")
}

def process_stationarity(s_id, serie):
    s_transformed, steps, status = make_stationary(serie)

    return {
        "unique_id": s_id,
        "transformations": steps,
        "n_transformations": len(steps),
        "final_status": status
    }

stationarity_results = Parallel(n_jobs=-1, verbose=10)(
    delayed(process_stationarity)(s_id, serie)
    for s_id, serie in grouped_series.items()
)

stationarity_df = pd.DataFrame(stationarity_results)

final_df = final_df.merge(
    stationarity_df,
    on="unique_id",
    how="left"
).sort_values(
    by="unique_id",
    key=lambda col: col.str.extract(r"(\d+)")[0].astype(int)
).reset_index(drop=True)

final_df.head()

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.19407238936402588s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.029224157333374023s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done 110 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done 144 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.06209087371826172s.) Setting batch_size=8.
[Parallel(n_jo

,unique_id,size,complexity,transformations,n_transformations,final_status
0,M80,585,22292.093,[],0,stationary
1,M160,91,273.198,[difference],1,stationary
2,M251,289,2168.225,[difference],1,stationary
3,M293,150,5295.457,[detrend],1,stationary
4,M591,142,372.494,[difference],1,stationary


## 7. ACF and PACF visual inspection

ARIMA is included as a classical statistical model because it can capture autocorrelation, trend, and seasonality through parameterised structures. ACF and PACF plots are exported for stationary series to provide diagnostic visual evidence of temporal dependence.

**Report reference:** Section 2.2 (*Forecasting Models*).

In [18]:
os.makedirs("acf_pacf_stationary", exist_ok=True)

stationary_series_ids = final_df.loc[ final_df["final_status"] == "stationary", "unique_id" ].tolist()

batch_size = 100
n_series = len(stationary_series_ids)

for start in range(0, n_series, batch_size):

    end = min(start + batch_size, n_series)
    subset_ids = stationary_series_ids[start:end]

    n_series_per_row = 2
    n_cols = 2 * n_series_per_row
    n_rows = int(np.ceil(len(subset_ids) / n_series_per_row))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows))
    axes = np.array(axes).flatten()

    plot_idx = 0

    for s_id in subset_ids:

        serie = df[df["unique_id"] == s_id].sort_values("ds")["y"]
        lags = min(30, len(serie) // 2)

        #ACF
        plot_acf(serie, ax=axes[plot_idx], lags=lags)
        axes[plot_idx].set_title(f"{s_id} - ACF")

        #PACF
        plot_pacf(serie, ax=axes[plot_idx + 1], lags=lags, method='ywm')
        axes[plot_idx + 1].set_title(f"{s_id} - PACF")

        plot_idx += 2

    for j in range(plot_idx, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()

    first_id = subset_ids[0]
    last_id = subset_ids[-1]

    filename = f"acf_pacf_stationary/acf_pacf_{first_id}_{last_id}.png"

    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()

## 8. Prepare the forecasting data format

The experimental setup requires that each model is fitted independently for each series while preserving temporal order. This cell converts the time index to integer format and sorts the observations by series identifier and time.

**Report reference:** Section 3.2 (*Experimental Setup*).

In [19]:
df_series = df.copy()

df_series["ds"] = df_series["ds"].astype("int64")

df_series = df_series.sort_values(
    by=["unique_id", "ds"],
    key=lambda col: col.str.extract(r"(\d+)")[0].astype(int)
              if col.name == "unique_id" else col
).reset_index(drop=True)

df_series

,unique_id,ds,y
0,M80,1,4580.0
1,M80,2,4260.0
2,M80,3,4430.0
3,M80,4,3400.0
4,M80,5,3300.0
...,...,...,...
95748,M47801,75,2364.0
95749,M47801,76,2775.2
95750,M47801,77,1554.2
95751,M47801,78,1096.3


## 9. Define the AutoARIMA candidate models

For each time series, two AutoARIMA specifications are considered: a seasonal model with `season_length=12`, corresponding to annual seasonality in monthly data, and a non-seasonal model with `season_length=1`. Both candidates are limited to at most two non-seasonal differences.

**Report reference:** Section 3.2 (*Models*).

In [23]:
models = [
    AutoARIMA(
        stepwise=False,
        approximation=False,
        season_length=12,
        alias="auto_seasonal",
        max_d=2,
        max_D=0
    ),
    AutoARIMA(
        stepwise=False,
        approximation=False,
        season_length=1,
        alias="auto_no_season",
        max_d=2,
        max_D=0
    ),
]

## 10. Train/test split

The final 18 observations of each monthly series are reserved as the test set, corresponding to an 18-month forecasting horizon. All previous observations are used for training.

**Report reference:** Section 3.2 (*Experimental Setup*).

In [24]:
test_size = 18
train_df = df_series.groupby("unique_id").head(-test_size)
test_df = df_series.groupby("unique_id").tail(test_size)

## 11. Growing-window cross-validation on the training set

Growing-window cross-validation procedure on the training set. At each validation window, the model is trained on an expanding initial segment and evaluated on the following unseen block, preserving the temporal order of observations. This cell applies the procedure with `h=18`, `step_size=1`, and `n_windows=6`.

**Report references:** Section 3.2 (*Experimental Setup*) and Appendix A, Figure 7.

In [26]:
def run_cv_series(uid, group_data):
    sf = StatsForecast(models=models, freq=1, n_jobs=1)
    result = sf.cross_validation(
        df=group_data,
        h=18,
        step_size=1,
        n_windows=6
    ).reset_index(drop=True)

    return result

results = Parallel(
    n_jobs=-1, 
    verbose=10
)(
    delayed(run_cv_series)(uid, group.reset_index(drop=True))
    for uid, group in train_df.groupby("unique_id")

)

cv_df_cv = pd.concat(results, ignore_index=True)
cv_df_cv.to_csv("cv_df_cv.csv", index=False)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:   32.3s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:  3.2min
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:  4.0min
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed:  4.5min
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:  4.8min
[Parallel(n_jobs=-1)]: Done 105 tasks      | elapsed:  6.0min
[Parallel(n_jobs=-1)]: Done 122 tasks      | elapsed:  7.3min
[Parallel(n_jobs=-1)]: Done 141 tasks      | elapsed:  8.5min
[Parallel(n_jobs=-1)]: Done 160 tasks      | elapsed:  9.9min
[Parallel(n_jobs=-1)]: Done 181 tasks      | elapsed: 11.2min
[Parallel(n_jobs=-1)]: Done 202 tasks      | elapsed: 12.6min
[Parallel(n_jobs=-1)]: Done 225 tasks      | elapsed: 1

## 12. Compute cross-validation errors

Forecasting performance is evaluated with NMAE, NRMSE, and SMAPE. This cell computes those metrics from the cross-validation forecasts for the seasonal and non-seasonal AutoARIMA candidates.

**Report reference:** Section 3.2 (*Evaluation Metrics*), Equations 5–7.

In [27]:
def nmae(y_true, y_pred):
    mean_true = np.mean(y_true)
    if mean_true == 0:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / mean_true

def nrmse(y_true, y_pred):
    mean_true = np.mean(y_true)
    if mean_true == 0:
        return np.nan
    return np.sqrt(np.mean((y_true - y_pred) ** 2)) / mean_true

def smape(y_true, y_pred):
    denominators = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominators != 0
    if not np.any(mask):
        return np.nan
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denominators[mask]) * 100

results = []
models_alias = ["auto_seasonal", "auto_no_season"]

for uid, group in cv_df_cv.groupby("unique_id"):
    for model in models_alias:
        group_clean = group.dropna(subset=["y", model])

        y_true = group_clean["y"].values
        y_pred = group_clean[model].values

        results.append({
            "unique_id": uid,
            "model": model,
            "NMAE":  nmae(y_true, y_pred),
            "NRMSE": nrmse(y_true, y_pred),
            "SMAPE": smape(y_true, y_pred)
        })

metrics_df = pd.DataFrame(results)
metrics_df = metrics_df.sort_values(
    by=["unique_id", "NMAE"],
    ascending=[True, True]
).reset_index(drop=True)

metrics_df.head()

,unique_id,model,NMAE,NRMSE,SMAPE
0,M10019,auto_seasonal,0.022343,0.037086,2.216915
1,M10019,auto_no_season,0.022343,0.037086,2.216915
2,M10283,auto_no_season,0.108438,0.127827,11.442250
3,M10283,auto_seasonal,0.117850,0.141050,12.748674
4,M10410,auto_seasonal,0.132291,0.165035,13.188974


## 13. Select the best AutoARIMA specification per series

The seasonal or non-seasonal AutoARIMA specification is selected independently for each series according to cross-validation performance. The primary criterion is SMAPE, followed by NRMSE and then NMAE as tie-breakers.

**Report reference:** Section 3.2 (*Models*).

In [28]:
best_models = (
    metrics_df
    .sort_values(["unique_id", "SMAPE", "NRMSE", "NMAE"])
    .drop_duplicates("unique_id")
)
 
best_models.head()

,unique_id,model,NMAE,NRMSE,SMAPE
0,M10019,auto_seasonal,0.022343,0.037086,2.216915
2,M10283,auto_no_season,0.108438,0.127827,11.442250
4,M10410,auto_seasonal,0.132291,0.165035,13.188974
6,M10428,auto_seasonal,0.006647,0.008466,0.661052
8,M10505,auto_no_season,0.120546,0.173893,11.971029


## 14. Fit the selected AutoARIMA model and forecast the test horizon

After model selection, the chosen AutoARIMA specification is fitted to the full training set of each series and used to forecast the final 18 observations.

**Report reference:** Section 3.2 (*Models*).

In [29]:
def process_series(row):
    uid = row["unique_id"]
    model_alias = row["model"]
    
    train = train_df[train_df["unique_id"] == uid].sort_values("ds").copy()
    test  = test_df[test_df["unique_id"] == uid].sort_values("ds").copy()
    
    m = 12 if model_alias == "auto_seasonal" else 1 #choose according to Cross-Validation results

    sf = StatsForecast(
        models=[AutoARIMA(
            season_length=m,
            stepwise=False,
            approximation=False,
            max_d=2,
            max_D=0
        )],
        freq=1,
        n_jobs=1
    )
    sf.fit(train[["unique_id", "ds", "y"]])

    result = sf.predict(h=18)

    return result

results = Parallel(n_jobs=-1, verbose=10)(
    delayed(process_series)(row)
    for _, row in best_models.iterrows()
)

forecast_df = pd.concat(results).reset_index(drop=True)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    9.9s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:   13.9s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:   16.0s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:   24.0s
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:   29.2s
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed:   32.8s
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:   36.3s
[Parallel(n_jobs=-1)]: Done 105 tasks      | elapsed:   44.5s
[Parallel(n_jobs=-1)]: Done 122 tasks      | elapsed:   54.3s
[Parallel(n_jobs=-1)]: Done 141 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 160 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 181 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done 202 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done 225 tasks      | elapsed:  

## 15. Align forecasts with test observations

The forecasts are merged with the held-out observations so that final errors are computed only on the 18-month test horizon defined by the experimental setup.

**Report reference:** Section 3.2 (*Experimental Setup*).

In [30]:
forecast_df = forecast_df.drop(columns=["y"], errors="ignore").merge(
    test_df[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="left"
)
forecast_df.head()

,unique_id,ds,AutoARIMA,y
0,M10019,329,990.342277,989.754
1,M10019,330,990.342277,998.934
2,M10019,331,990.342277,1013.999
3,M10019,332,990.342277,1029.607
4,M10019,333,990.342277,1041.212


## 16. Evaluate AutoARIMA on the test set

The final AutoARIMA forecasts are evaluated using the same three error metrics defined in the report: NMAE, NRMSE, and SMAPE. The resulting values are merged into `final_df` for later comparison with the baselines and machine-learning models.

**Report reference:** Section 3.2 (*Evaluation Metrics*).

In [31]:
def nmae(df, models, id_col='unique_id', target_col='y', **kwargs):
    result = {}
    for model in models:
        vals = df.groupby(id_col).apply(
            lambda g: (
                np.mean(np.abs(g[target_col].values - g[model].values)) / np.mean(g[target_col].values)
                if np.mean(g[target_col].values) != 0 else np.nan
            ),
            include_groups=False
        )
        result[model] = vals
    return pd.DataFrame(result).rename_axis(id_col).reset_index()

def nrmse(df, models, id_col='unique_id', target_col='y', **kwargs):
    result = {}
    for model in models:
        vals = df.groupby(id_col).apply(
            lambda g: (
                np.sqrt(np.mean((g[target_col].values - g[model].values) ** 2)) / np.mean(g[target_col].values)
                if np.mean(g[target_col].values) != 0 else np.nan
            ),
            include_groups=False
        )
        result[model] = vals
    return pd.DataFrame(result).rename_axis(id_col).reset_index()

results = evaluate(
    forecast_df,
    metrics=[nmae, nrmse, ufl.smape],
    train_df=train_df,
)

metrics_df = results.pivot(index='unique_id', columns='metric', values='AutoARIMA')
metrics_df.columns = [f"{col}_AutoARIMA" for col in metrics_df.columns]
metrics_df = metrics_df.reset_index()

patterns = ["nmae_AutoARIMA", "nrmse_AutoARIMA", "smape_AutoARIMA"]
cols_to_drop = [c for c in final_df.columns if any(c.startswith(p) for p in patterns)]
final_df = final_df.drop(columns=cols_to_drop, errors='ignore')

final_df = final_df.merge(metrics_df, on='unique_id', how='left')
final_df.head()

,unique_id,size,complexity,transformations,n_transformations,final_status,nmae_AutoARIMA,nrmse_AutoARIMA,smape_AutoARIMA
0,M80,585,22292.093,[],0,stationary,0.119518,0.154520,0.060977
1,M160,91,273.198,[difference],1,stationary,0.153033,0.178445,0.080407
2,M251,289,2168.225,[difference],1,stationary,0.599868,0.628307,0.234093
3,M293,150,5295.457,[detrend],1,stationary,0.278123,0.306875,0.155947
4,M591,142,372.494,[difference],1,stationary,0.043832,0.049612,0.021588


## 17. Generate Naive and Seasonal Naive baselines

Naive and Seasonal Naive are included as simple but important baseline models. They provide reference performance levels for assessing whether more complex forecasting models are actually useful.

**Report references:** Sections 2.2 (*Forecasting Models*) and 3.2 (*Models*).

In [32]:
all_baseline_forecasts = []

for uid in train_df["unique_id"].unique():
    
    train = train_df[train_df["unique_id"] == uid].sort_values("ds").copy()
    test  = test_df[test_df["unique_id"] == uid].sort_values("ds").copy()
    
    history = train["y"].tolist()
    
    naive_preds = []
    
    for i in range(len(test)):
        naive_preds.append(history[-1])
    
    sf_baseline = StatsForecast(models=[SeasonalNaive(season_length=12)], freq=1, n_jobs=1)
    sf_baseline.fit(train[["unique_id", "ds", "y"]])
    snaive_preds = sf_baseline.predict(h=test_size)["SeasonalNaive"].tolist()
    
    result = test.copy()
    result["naive"] = naive_preds
    result["seasonal_naive"] = snaive_preds
    
    all_baseline_forecasts.append(result)

baseline_df = pd.concat(all_baseline_forecasts).reset_index(drop=True)
baseline_df.head()

,unique_id,ds,y,naive,seasonal_naive
0,M80,568,4630.0,5690.0,5150.0
1,M80,569,2800.0,5690.0,4070.0
2,M80,570,4520.0,5690.0,4930.0
3,M80,571,4740.0,5690.0,4130.0
4,M80,572,4060.0,5690.0,3800.0


## 18. Plot ARIMA and baseline forecasts

These diagnostic plots compare the training observations, the held-out test observations, the selected AutoARIMA forecasts, and the two baseline forecasts. They provide qualitative support for interpreting the quantitative error results.

**Report reference:** Section 4 (*Results and Analysis*).

In [33]:
os.makedirs("plots_forecasts_arima", exist_ok=True)

batch_size = 100
n_series = len(series_ids)

colors = {
    "train": "#1f77b4",
    "test": "#ff7f0e",
    "arima": "#2ca02c",
    "naive": "#d62728",
    "snaive": "#9467bd",
}

zoom_window = test_size + 12

for start in range(0, n_series, batch_size):
    
    end = min(start + batch_size, n_series)
    subset_ids = series_ids[start:end]
    
    n_rows = len(subset_ids)
    n_cols = 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows))
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for idx, uid in enumerate(subset_ids):
        row = idx

        train_uid = train_df[train_df["unique_id"] == uid]
        test_uid  = test_df[test_df["unique_id"] == uid]
        base_uid  = baseline_df[baseline_df["unique_id"] == uid]
        pred_uid  = forecast_df[forecast_df["unique_id"] == uid]

        #série completa
        ax_full = axes[row, 0]

        ax_full.plot(train_uid["ds"], train_uid["y"],
             color=colors["train"], linewidth=1.5, alpha=0.8,
             label="Train", zorder=1)

        ax_full.plot(test_uid["ds"], test_uid["y"],
             color=colors["test"], linewidth=2.5,
             label="Test", zorder=2)

        ax_full.plot(pred_uid["ds"], pred_uid["AutoARIMA"],
             color=colors["arima"], linewidth=2,
             label="ARIMA", zorder=3)

        ax_full.plot(base_uid["ds"], base_uid["naive"],
             color=colors["naive"], linestyle="--",
             linewidth=1, alpha=0.8,
             label="Naive", zorder=4)

        ax_full.plot(base_uid["ds"], base_uid["seasonal_naive"],
             color=colors["snaive"], linestyle=":",
             linewidth=1, alpha=0.8,
             label="Seasonal Naive", zorder=5)

        ax_full.axvline(x=test_uid["ds"].iloc[0], color="black", linestyle="dashed", linewidth=0.8)
        ax_full.set_title(f"{uid} - Full", fontsize=9)
        ax_full.legend(fontsize=7)
        ax_full.tick_params(labelsize=7)

        #zoom
        ax_zoom = axes[row, 1]

        zoom_df = pd.concat([train_uid, test_uid]).iloc[-zoom_window:]

        ax_zoom.plot(zoom_df["ds"], zoom_df["y"],
             color=colors["train"], linewidth=2.5,
             label="Real", zorder=1)

        ax_zoom.plot(pred_uid["ds"], pred_uid["AutoARIMA"],
             color=colors["arima"], linewidth=2,
             label="ARIMA", zorder=2)

        ax_zoom.plot(base_uid["ds"], base_uid["naive"],
             color=colors["naive"], linestyle="--",
             linewidth=1.5, alpha=0.8,
             label="Naive", zorder=3)

        ax_zoom.plot(base_uid["ds"], base_uid["seasonal_naive"],
             color=colors["snaive"], linestyle=":",
             linewidth=1.5, alpha=0.8,
             label="Seasonal Naive", zorder=4)

        ax_zoom.axvline(x=test_uid["ds"].iloc[0], color="black", linestyle="dashed", linewidth=0.8)
        ax_zoom.set_title(f"{uid} - Zoom", fontsize=9)
        ax_zoom.legend(fontsize=7)
        ax_zoom.tick_params(labelsize=7)

    plt.tight_layout()

    first_id = subset_ids[0]
    last_id  = subset_ids[-1]

    filename = f"plots_forecasts_arima/plot_{first_id}_{last_id}.png"

    plt.savefig(filename, dpi=100, bbox_inches="tight")
    plt.close()

## 19. Evaluate the baseline models

The Naive and Seasonal Naive forecasts are evaluated with NMAE, NRMSE, and SMAPE, using the same definitions applied to AutoARIMA. The resulting baseline errors are stored in `final_df` to support model comparisons.

**Report reference:** Section 3.2 (*Evaluation Metrics*).

In [34]:
results_naive = evaluate(
    baseline_df,
    metrics=[nmae, nrmse, ufl.smape],
    train_df=train_df,
)

metrics_naive = results_naive.pivot(index='unique_id', columns='metric', values=['naive', 'seasonal_naive'])
metrics_naive.columns = [f"{metric}_{model}" for model, metric in metrics_naive.columns]
metrics_naive = metrics_naive.reset_index()

patterns = ["nmae_naive", "nrmse_naive", "smape_naive",
            "nmae_seasonal_naive", "nrmse_seasonal_naive", "smape_seasonal_naive"]
cols_to_drop = [c for c in final_df.columns if any(c.startswith(p) for p in patterns)]
final_df = final_df.drop(columns=cols_to_drop, errors='ignore')

final_df = final_df.merge(metrics_naive, on='unique_id', how='left')
final_df.head()

,unique_id,size,complexity,transformations,n_transformations,final_status,nmae_AutoARIMA,nrmse_AutoARIMA,smape_AutoARIMA,nmae_naive,nrmse_naive,smape_naive,nmae_seasonal_naive,nrmse_seasonal_naive,smape_seasonal_naive
0,M80,585,22292.093,[],0,stationary,0.119518,0.154520,0.060977,0.323598,0.352719,0.143624,0.143448,0.169693,0.071112
1,M160,91,273.198,[difference],1,stationary,0.153033,0.178445,0.080407,0.148942,0.173730,0.078029,0.151324,0.176683,0.079474
2,M251,289,2168.225,[difference],1,stationary,0.599868,0.628307,0.234093,0.415595,0.440782,0.176457,0.779170,0.840742,0.271204
3,M293,150,5295.457,[detrend],1,stationary,0.278123,0.306875,0.155947,0.248370,0.286489,0.135658,0.232967,0.258599,0.133951
4,M591,142,372.494,[difference],1,stationary,0.043832,0.049612,0.021588,0.028322,0.030695,0.014078,0.031077,0.036928,0.015414


## 20. Relative improvement against baselines

Because the baselines are used as reference levels, this cell computes the percentage improvement of AutoARIMA over Naive and Seasonal Naive for each evaluation metric. These columns help quantify whether the statistical model improves over simple benchmark forecasts.

**Report references:** Sections 2.2 (*Forecasting Models*) and 4 (*Results and Analysis*).

In [36]:
def safe_pct(num, den):
    with np.errstate(divide="ignore", invalid="ignore"):
        result = np.where(
            den == 0,
            np.nan,
            ((den - num) / den) * 100
        )
    return result

# vs seasonal_naive
final_df["%nmae_arima_vs_snaive"]  = safe_pct(final_df["nmae_AutoARIMA"],  final_df["nmae_seasonal_naive"])
final_df["%nrmse_arima_vs_snaive"] = safe_pct(final_df["nrmse_AutoARIMA"], final_df["nrmse_seasonal_naive"])
final_df["%smape_arima_vs_snaive"] = safe_pct(final_df["smape_AutoARIMA"], final_df["smape_seasonal_naive"])

# vs naive
final_df["%nmae_arima_vs_naive"]   = safe_pct(final_df["nmae_AutoARIMA"],  final_df["nmae_naive"])
final_df["%nrmse_arima_vs_naive"]  = safe_pct(final_df["nrmse_AutoARIMA"], final_df["nrmse_naive"])
final_df["%smape_arima_vs_naive"]  = safe_pct(final_df["smape_AutoARIMA"], final_df["smape_naive"])

# NaN for string
pct_cols = [col for col in final_df.columns if col.startswith("%")]
for col in pct_cols:
    final_df[col] = final_df[col].apply(
        lambda x: "Division by Zero" if pd.isna(x) else x
    )

final_df.to_csv("final_df.csv", index=False)
final_df.head()

,unique_id,size,complexity,transformations,n_transformations,final_status,nmae_AutoARIMA,nrmse_AutoARIMA,smape_AutoARIMA,nmae_naive,...,smape_naive,nmae_seasonal_naive,nrmse_seasonal_naive,smape_seasonal_naive,%nmae_arima_vs_snaive,%nrmse_arima_vs_snaive,%smape_arima_vs_snaive,%nmae_arima_vs_naive,%nrmse_arima_vs_naive,%smape_arima_vs_naive
0,M80,585,22292.093,[],0,stationary,0.119518,0.154520,0.060977,0.323598,...,0.143624,0.143448,0.169693,0.071112,16.682007,8.941047,14.252458,63.065906,56.191667,57.544181
1,M160,91,273.198,[difference],1,stationary,0.153033,0.178445,0.080407,0.148942,...,0.078029,0.151324,0.176683,0.079474,-1.129450,-0.997299,-1.173563,-2.746526,-2.714051,-3.046716
2,M251,289,2168.225,[difference],1,stationary,0.599868,0.628307,0.234093,0.415595,...,0.176457,0.779170,0.840742,0.271204,23.011890,25.267502,13.683699,-44.339754,-42.543653,-32.662704
3,M293,150,5295.457,[detrend],1,stationary,0.278123,0.306875,0.155947,0.248370,...,0.135658,0.232967,0.258599,0.133951,-19.383345,-18.668373,-16.421203,-11.979653,-7.115953,-14.956206
4,M591,142,372.494,[difference],1,stationary,0.043832,0.049612,0.021588,0.028322,...,0.014078,0.031077,0.036928,0.015414,-41.046946,-34.349389,-40.052131,-54.762140,-61.631340,-53.341109
